In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_absolute_error
import xgboost as xgb
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Create models directory
os.makedirs('models', exist_ok=True)
print("✅ Imports done")

✅ Imports done


In [2]:
# Load dataset
df = pd.read_csv('data/Sleep_health_and_lifestyle_dataset.csv')
print("Columns:", df.columns.tolist())
print(df.head())

Columns: ['Person ID', 'Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Blood Pressure', 'Heart Rate', 'Daily Steps', 'Sleep Disorder']
   Person ID Gender  Age            Occupation  Sleep Duration  \
0          1   Male   27     Software Engineer             6.1   
1          2   Male   28                Doctor             6.2   
2          3   Male   28                Doctor             6.2   
3          4   Male   28  Sales Representative             5.9   
4          5   Male   28  Sales Representative             5.9   

   Quality of Sleep  Physical Activity Level  Stress Level BMI Category  \
0                 6                       42             6   Overweight   
1                 6                       60             8       Normal   
2                 6                       60             8       Normal   
3                 4                       30             8        Obese   
4           

In [3]:
# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Cleaned columns:", df.columns.tolist())

# Map sleep disorder to fatigue label
def map_fatigue(row):
    sleep = row.get('sleep_duration', 7)
    quality = row.get('quality_of_sleep', 5)
    stress = row.get('stress_level', 5)
    
    if sleep < 5 or stress >= 8:
        return 'Overtrained'
    elif sleep < 6.5 or stress >= 6 or quality <= 4:
        return 'Fatigued'
    elif sleep >= 7.5 and stress <= 3 and quality >= 7:
        return 'Fresh'
    else:
        return 'Normal'

df['fatigue_label'] = df.apply(map_fatigue, axis=1)
print("\nFatigue distribution:")
print(df['fatigue_label'].value_counts())

Cleaned columns: ['person_id', 'gender', 'age', 'occupation', 'sleep_duration', 'quality_of_sleep', 'physical_activity_level', 'stress_level', 'bmi_category', 'blood_pressure', 'heart_rate', 'daily_steps', 'sleep_disorder']

Fatigue distribution:
fatigue_label
Normal         137
Fatigued        96
Fresh           71
Overtrained     70
Name: count, dtype: int64


In [4]:
# Select features
feature_cols = []
possible_cols = ['sleep_duration', 'quality_of_sleep', 'stress_level', 
                 'heart_rate', 'daily_steps', 'physical_activity_level']

for col in possible_cols:
    if col in df.columns:
        feature_cols.append(col)

print("Using features:", feature_cols)

# Prepare data
X = df[feature_cols].fillna(df[feature_cols].mean())
le = LabelEncoder()
y = le.fit_transform(df['fatigue_label'])

# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
fatigue_model = RandomForestClassifier(n_estimators=100, random_state=42)
fatigue_model.fit(X_train, y_train)

# Evaluate
y_pred = fatigue_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Fatigue Model Accuracy: {acc:.2%}")
print("Classes:", le.classes_)

# Save
joblib.dump(fatigue_model, 'models/fatigue_model.pkl')
joblib.dump(le, 'models/fatigue_label_encoder.pkl')
joblib.dump(feature_cols, 'models/fatigue_features.pkl')
print("✅ Fatigue model saved!")

Using features: ['sleep_duration', 'quality_of_sleep', 'stress_level', 'heart_rate', 'daily_steps', 'physical_activity_level']

✅ Fatigue Model Accuracy: 100.00%
Classes: ['Fatigued' 'Fresh' 'Normal' 'Overtrained']
✅ Fatigue model saved!


In [5]:
np.random.seed(42)
n = 2000

# Generate realistic athlete data
training_load = np.random.uniform(100, 800, n)
prev_load = np.random.uniform(100, 700, n)
sleep_hours = np.random.uniform(4, 9, n)
soreness = np.random.randint(1, 10, n)
pain_level = np.random.randint(0, 8, n)
consecutive_days = np.random.randint(0, 10, n)
recovery_score = np.random.uniform(20, 100, n)

# Calculate load spike
load_spike = ((training_load - prev_load) / np.maximum(prev_load, 1)) * 100

# Create injury risk label based on realistic rules
def calculate_risk(i):
    score = 0
    if load_spike[i] > 30: score += 2
    if load_spike[i] > 50: score += 1
    if sleep_hours[i] < 6: score += 1
    if sleep_hours[i] < 5: score += 1
    if soreness[i] >= 7: score += 2
    if pain_level[i] >= 5: score += 2
    if consecutive_days[i] >= 6: score += 1
    if recovery_score[i] < 40: score += 1
    
    if score >= 5: return 2    # High
    elif score >= 3: return 1  # Medium
    else: return 0             # Low

risk_labels = np.array([calculate_risk(i) for i in range(n)])

injury_df = pd.DataFrame({
    'training_load': training_load,
    'prev_load': prev_load,
    'load_spike_pct': load_spike,
    'sleep_hours': sleep_hours,
    'soreness': soreness,
    'pain_level': pain_level,
    'consecutive_days': consecutive_days,
    'recovery_score': recovery_score,
    'risk': risk_labels
})

print("Injury Risk Distribution:")
print(injury_df['risk'].value_counts())
print("\nSample data:")
print(injury_df.head())

Injury Risk Distribution:
risk
2    749
1    663
0    588
Name: count, dtype: int64

Sample data:
   training_load   prev_load  load_spike_pct  sleep_hours  soreness  \
0     362.178083  257.023410       40.912488     6.859979         1   
1     765.500014  248.187279      208.436442     8.027162         1   
2     612.395759  643.752748       -4.870968     7.800805         3   
3     519.060939  249.727720      107.850750     4.769500         5   
4     209.213048  263.169836      -20.502649     4.746247         3   

   pain_level  consecutive_days  recovery_score  risk  
0           0                 9       46.643139     1  
1           2                 4       51.996970     1  
2           2                 0       53.866138     0  
3           4                 5       70.094356     2  
4           3                 9       42.943658     1  


In [6]:
injury_features = ['training_load', 'prev_load', 'load_spike_pct', 
                   'sleep_hours', 'soreness', 'pain_level', 
                   'consecutive_days', 'recovery_score']

X = injury_df[injury_features]
y = injury_df['risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

injury_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)
injury_model.fit(X_train, y_train)

y_pred = injury_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"✅ Injury Risk Model Accuracy: {acc:.2%}")

# Save
joblib.dump(injury_model, 'models/injury_model.pkl')
joblib.dump(injury_features, 'models/injury_features.pkl')
print("✅ Injury model saved!")

✅ Injury Risk Model Accuracy: 98.00%
✅ Injury model saved!


In [7]:
# Load body fat dataset
bf_df = pd.read_csv('data/bodyfat.csv')
print("Columns:", bf_df.columns.tolist())
print(bf_df.head())

Columns: ['Density', 'BodyFat', 'Age', 'Weight', 'Height', 'Neck', 'Chest', 'Abdomen', 'Hip', 'Thigh', 'Knee', 'Ankle', 'Biceps', 'Forearm', 'Wrist']
   Density  BodyFat  Age  Weight  Height  Neck  Chest  Abdomen    Hip  Thigh  \
0   1.0708     12.3   23  154.25   67.75  36.2   93.1     85.2   94.5   59.0   
1   1.0853      6.1   22  173.25   72.25  38.5   93.6     83.0   98.7   58.7   
2   1.0414     25.3   22  154.00   66.25  34.0   95.8     87.9   99.2   59.6   
3   1.0751     10.4   26  184.75   72.25  37.4  101.8     86.4  101.2   60.1   
4   1.0340     28.7   24  184.25   71.25  34.4   97.3    100.0  101.9   63.2   

   Knee  Ankle  Biceps  Forearm  Wrist  
0  37.3   21.9    32.0     27.4   17.1  
1  37.3   23.4    30.5     28.9   18.2  
2  38.9   24.0    28.8     25.2   16.6  
3  37.3   22.8    32.4     29.4   18.2  
4  42.2   24.0    32.2     27.7   17.7  


In [8]:
# Clean column names
bf_df.columns = bf_df.columns.str.strip().str.lower()
print("Cleaned columns:", bf_df.columns.tolist())

# Find body fat percentage column
bf_col = None
for col in bf_df.columns:
    if 'fat' in col.lower() or 'bodyfat' in col.lower() or 'body' in col.lower():
        bf_col = col
        break

if bf_col is None:
    bf_col = bf_df.columns[0]

print(f"Target column: {bf_col}")

# Select features — use whatever measurement columns exist
possible_features = ['age', 'weight', 'height', 'neck', 'chest', 
                     'abdomen', 'hip', 'thigh', 'knee', 'ankle',
                     'biceps', 'forearm', 'wrist', 'bmi',
                     'density', 'adiposity']

bf_features = [col for col in possible_features if col in bf_df.columns]
print(f"Using features: {bf_features}")

# Remove outliers
bf_df = bf_df[bf_df[bf_col] > 2]
bf_df = bf_df[bf_df[bf_col] < 50]

X = bf_df[bf_features].fillna(bf_df[bf_features].mean())
y = bf_df[bf_col]

print(f"\nDataset size: {len(X)}")
print(f"Body fat range: {y.min():.1f}% - {y.max():.1f}%")

Cleaned columns: ['density', 'bodyfat', 'age', 'weight', 'height', 'neck', 'chest', 'abdomen', 'hip', 'thigh', 'knee', 'ankle', 'biceps', 'forearm', 'wrist']
Target column: bodyfat
Using features: ['age', 'weight', 'height', 'neck', 'chest', 'abdomen', 'hip', 'thigh', 'knee', 'ankle', 'biceps', 'forearm', 'wrist', 'density']

Dataset size: 250
Body fat range: 3.0% - 47.5%


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

bf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42
)
bf_model.fit(X_train, y_train)

y_pred = bf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"✅ Body Fat Model MAE: {mae:.2f}%")
print(f"Sample predictions: {y_pred[:5].round(1)}")
print(f"Actual values:      {y_test.values[:5].round(1)}")

# Save
joblib.dump(bf_model, 'models/bodyfat_model.pkl')
joblib.dump(bf_features, 'models/bodyfat_features.pkl')
print("✅ Body fat model saved!")

✅ Body Fat Model MAE: 0.16%
Sample predictions: [23.  19.2 11.4 26.  22.2]
Actual values:      [23.3 19.2 11.3 26.1 22.1]
✅ Body fat model saved!


In [10]:
import os

models = [
    'models/fatigue_model.pkl',
    'models/fatigue_label_encoder.pkl', 
    'models/fatigue_features.pkl',
    'models/injury_model.pkl',
    'models/injury_features.pkl',
    'models/bodyfat_model.pkl',
    'models/bodyfat_features.pkl'
]

print("Model files:")
for m in models:
    exists = os.path.exists(m)
    size = os.path.getsize(m) / 1024 if exists else 0
    print(f"  {'✅' if exists else '❌'} {m} ({size:.1f} KB)")

print("\n🎉 All models trained and saved!")
print("Next step: Copy .pkl files to backend/ml/models/")

Model files:
  ✅ models/fatigue_model.pkl (241.6 KB)
  ✅ models/fatigue_label_encoder.pkl (0.5 KB)
  ✅ models/fatigue_features.pkl (0.1 KB)
  ✅ models/injury_model.pkl (456.3 KB)
  ✅ models/injury_features.pkl (0.1 KB)
  ✅ models/bodyfat_model.pkl (1501.2 KB)
  ✅ models/bodyfat_features.pkl (0.1 KB)

🎉 All models trained and saved!
Next step: Copy .pkl files to backend/ml/models/
